# 스트레스 점수 예측 V19 — Pair + Triple Neighbor

**Repeated-CV 2D/3D Neighbor Ensemble**

- Public 0.1272333의 2D Pair-Neighbor를 기준 모델로 사용
- 8개 핵심 변수의 28개 2D 조합과 56개 3D 조합에서 Train 1-NN 탐색
- 2D·3D 예측의 분위수와 혼합 비율 비교
- 3개의 서로 다른 5-Fold 분할에서 모두 기준 모델을 이긴 조합만 선택
- 각 Fold의 중앙값·순위·이웃 탐색은 Fold Train으로만 학습

In [10]:
from pathlib import Path
import os
from itertools import combinations

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [11]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")
TARGET = "stress_score"
ID_COLUMN = "ID"
N_ESTIMATORS = 1200
MODEL_SEED = 42
CV_SEEDS = (42, 77, 2026)
TREE_QUANTILE = 0.52

NEIGHBOR_QUANTILES = (0.48, 0.50, 0.52, 0.54)
PAIR_WEIGHTS = (0.10, 0.15, 0.20)
TRIPLE_WEIGHTS = (0.00, 0.05, 0.10, 0.15, 0.20)
ROUND_OPTIONS = (False, True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2,
    "height": 2, "glucose": 3, "weight": 2,
    "cholesterol_glucose_ratio": 2, "bone_density": 4,
}
NEIGHBOR_COLUMNS = list(FEATURE_COPY_COUNTS)

## 2. 데이터와 파생변수

In [12]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("ID 순서가 다릅니다.")

def add_features(frame):
    r=frame.copy(); hm=r["height"]/100
    sbp=r["systolic_blood_pressure"]; dbp=r["diastolic_blood_pressure"]
    r["bmi"]=r["weight"]/hm.pow(2)
    r["pulse_pressure"]=sbp-dbp
    r["mean_arterial_pressure"]=(sbp+2*dbp)/3
    r["blood_pressure_ratio"]=sbp/(dbp+1e-6)
    r["cholesterol_glucose_ratio"]=r["cholesterol"]/(r["glucose"]+1e-6)
    r["missing_count"]=r.isna().sum(axis=1)
    return r

def add_copies(frame):
    r=frame.copy()
    for f,n in FEATURE_COPY_COUNTS.items():
        for i in range(1,n+1): r[f"{f}__copy{i}"]=r[f]
    return r

y=train[TARGET].reset_index(drop=True)
X=add_features(train.drop(columns=[TARGET,ID_COLUMN])).reset_index(drop=True)
X_test=add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
print("Train:",X.shape,"| 2D + 3D combinations:",len(list(combinations(NEIGHBOR_COLUMNS,2))), "+", len(list(combinations(NEIGHBOR_COLUMNS,3))))

Train: (3000, 22) | 2D + 3D combinations: 28 + 56


## 3. ExtraTrees와 Train-기준 순위 함수

In [13]:
def make_preprocessor(frame):
    cat=frame.select_dtypes(include=["object","category","string"]).columns.tolist()
    num=frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("cat",Pipeline([
            ("imp",SimpleImputer(strategy="constant",fill_value="missing")),
            ("oh",OneHotEncoder(handle_unknown="ignore",sparse_output=False)),
        ]),cat),
        ("num",SimpleImputer(strategy="median",add_indicator=True),num),
    ])

def tree_predict(train_x, valid_x, target):
    wt,wv=add_copies(train_x),add_copies(valid_x)
    p=make_preprocessor(wt); a=p.fit_transform(wt); b=p.transform(wv)
    m=ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,min_samples_leaf=1,max_features=1,
        random_state=MODEL_SEED,n_jobs=-1,
    )
    m.fit(a,target)
    matrix=np.column_stack([t.predict(b) for t in m.estimators_])
    return np.clip(np.round(np.quantile(matrix,TREE_QUANTILE,axis=1),2),0,1)

def rank_transform(train_values, valid_values):
    median=np.nanmedian(train_values)
    a=np.nan_to_num(train_values,nan=median)
    b=np.nan_to_num(valid_values,nan=median)
    sorted_a=np.sort(a)
    return (
        np.searchsorted(sorted_a,a,side="right")/len(a),
        np.searchsorted(sorted_a,b,side="right")/len(a),
    )

## 4. 2D·3D 1-NN 예측 소재 생성

In [14]:
def neighbor_materials(train_x, valid_x, target):
    train_rank={}; valid_rank={}
    for col in NEIGHBOR_COLUMNS:
        train_rank[col],valid_rank[col]=rank_transform(
            train_x[col].to_numpy(float),valid_x[col].to_numpy(float)
        )
    target_values=target.to_numpy(float)
    output={2:[],3:[]}
    for dimension in (2,3):
        for columns in combinations(NEIGHBOR_COLUMNS,dimension):
            distance=np.zeros((len(valid_x),len(train_x)),dtype=np.float32)
            for column in columns:
                distance += np.abs(
                    valid_rank[column][:,None]-train_rank[column][None,:]
                ).astype(np.float32)
            # 기준 V7과 동일한 np.argmin 동률 처리
            nearest=np.argmin(distance,axis=1)
            output[dimension].append(target_values[nearest])
    return {dimension:np.column_stack(values) for dimension,values in output.items()}

## 5. 3개 CV 분할에서 2D·3D OOF 생성

In [15]:
split_data={}
for cv_seed in CV_SEEDS:
    cv=KFold(5,shuffle=True,random_state=cv_seed)
    tree_oof=np.zeros(len(y))
    neighbor_oof={
        (dimension,quantile):np.zeros(len(y))
        for dimension in (2,3)
        for quantile in NEIGHBOR_QUANTILES
    }
    for fold,(ti,vi) in enumerate(cv.split(X),1):
        tree_oof[vi]=tree_predict(X.iloc[ti],X.iloc[vi],y.iloc[ti])
        materials=neighbor_materials(X.iloc[ti],X.iloc[vi],y.iloc[ti])
        for dimension in (2,3):
            for quantile in NEIGHBOR_QUANTILES:
                neighbor_oof[(dimension,quantile)][vi]=np.quantile(
                    materials[dimension],quantile,axis=1
                )
        print(f"CV {cv_seed} | Fold {fold}: complete")
    split_data[cv_seed]={"tree":tree_oof,"neighbor":neighbor_oof}
    baseline=np.clip(
        np.round(.85*tree_oof+.15*neighbor_oof[(2,.52)],2),0,1
    )
    print(f"CV {cv_seed} baseline MAE: {mean_absolute_error(y,baseline):.6f}")

CV 42 | Fold 1: complete
CV 42 | Fold 2: complete
CV 42 | Fold 3: complete
CV 42 | Fold 4: complete
CV 42 | Fold 5: complete
CV 42 baseline MAE: 0.148587
CV 77 | Fold 1: complete
CV 77 | Fold 2: complete
CV 77 | Fold 3: complete
CV 77 | Fold 4: complete
CV 77 | Fold 5: complete
CV 77 baseline MAE: 0.146393
CV 2026 | Fold 1: complete
CV 2026 | Fold 2: complete
CV 2026 | Fold 3: complete
CV 2026 | Fold 4: complete
CV 2026 | Fold 5: complete
CV 2026 baseline MAE: 0.144793


## 6. 모든 분할에서 안정적인 2D·3D 조합 선택

In [16]:
rows=[]
for pair_q in NEIGHBOR_QUANTILES:
 for triple_q in NEIGHBOR_QUANTILES:
  for pair_weight in PAIR_WEIGHTS:
   for triple_weight in TRIPLE_WEIGHTS:
    if pair_weight+triple_weight>=1: continue
    for rounding in ROUND_OPTIONS:
     scores=[]; improvements=[]
     for cv_seed in CV_SEEDS:
        item=split_data[cv_seed]
        baseline=np.clip(
            np.round(.85*item["tree"]+.15*item["neighbor"][(2,.52)],2),0,1
        )
        tree_weight=1-pair_weight-triple_weight
        pred=(
            tree_weight*item["tree"]
            +pair_weight*item["neighbor"][(2,pair_q)]
            +triple_weight*item["neighbor"][(3,triple_q)]
        )
        if rounding: pred=np.round(pred,2)
        pred=np.clip(pred,0,1)
        base_mae=mean_absolute_error(y,baseline); score=mean_absolute_error(y,pred)
        scores.append(score); improvements.append(base_mae-score)
     rows.append({
        "pair_quantile":pair_q,"triple_quantile":triple_q,
        "pair_weight":pair_weight,"triple_weight":triple_weight,
        "tree_weight":1-pair_weight-triple_weight,"rounding":rounding,
        "mean_CV_MAE":np.mean(scores),"mean_improvement":np.mean(improvements),
        "worst_improvement":min(improvements),
        "wins":sum(v>0 for v in improvements),
     })
results=pd.DataFrame(rows).sort_values(
    ["wins","worst_improvement","mean_CV_MAE"],ascending=[False,False,True]
).reset_index(drop=True)
eligible=results[
    (results.wins==len(CV_SEEDS))
    &(results.worst_improvement>=0.0001)
    &(results.triple_weight>0)
]
if len(eligible): selected=eligible.iloc[0]
else:
    selected=results[
        (results.pair_quantile==.52)&(results.pair_weight==.15)&
        (results.triple_weight==0)&(results.rounding)
    ].iloc[0]
print("Selected tree weight:",selected.tree_weight)
print("Selected 2D quantile / weight:",selected.pair_quantile,"/",selected.pair_weight)
print("Selected 3D quantile / weight:",selected.triple_quantile,"/",selected.triple_weight)
print("Round to 2 decimals:",bool(selected.rounding))
print("Repeated-CV mean MAE:",f"{selected.mean_CV_MAE:.6f}")
print("Worst split improvement:",f"{selected.worst_improvement:+.6f}")
print("Wins:",int(selected.wins),"/",len(CV_SEEDS))
results.head(20)

Selected tree weight: 0.8
Selected 2D quantile / weight: 0.5 / 0.1
Selected 3D quantile / weight: 0.48 / 0.1
Round to 2 decimals: True
Repeated-CV mean MAE: 0.146419
Worst split improvement: +0.000133
Wins: 3 / 3


,pair_quantile,triple_quantile,pair_weight,triple_weight,tree_weight,rounding,mean_CV_MAE,mean_improvement,worst_improvement,wins
0,0.50,0.48,0.10,0.10,0.80,True,0.146419,0.000172,0.000133,3
1,0.54,0.50,0.15,0.10,0.75,True,0.146431,0.000160,0.000123,3
2,0.54,0.48,0.10,0.10,0.80,True,0.146429,0.000162,0.000117,3
3,0.54,0.48,0.10,0.10,0.80,False,0.146463,0.000129,0.000116,3
4,0.50,0.50,0.10,0.10,0.80,True,0.146434,0.000157,0.000110,3
5,0.52,0.50,0.10,0.10,0.80,True,0.146464,0.000127,0.000110,3
6,0.54,0.48,0.10,0.15,0.75,False,0.146432,0.000160,0.000102,3
7,0.52,0.48,0.10,0.10,0.80,True,0.146423,0.000168,0.000093,3
8,0.48,0.54,0.10,0.10,0.80,True,0.146443,0.000148,0.000093,3
9,0.50,0.52,0.10,0.10,0.80,True,0.146461,0.000130,0.000090,3


## 7. 전체 Train으로 2D·3D Test 예측

In [17]:
pred_tree=tree_predict(X,X_test,y)
materials=neighbor_materials(X,X_test,y)
pred_pair=np.quantile(materials[2],float(selected.pair_quantile),axis=1)
pred_triple=np.quantile(materials[3],float(selected.triple_quantile),axis=1)
pred_v19=(
    float(selected.tree_weight)*pred_tree
    +float(selected.pair_weight)*pred_pair
    +float(selected.triple_weight)*pred_triple
)
if bool(selected.rounding): pred_v19=np.round(pred_v19,2)
pred_v19=np.clip(pred_v19,0,1)
print("Prediction range:",pred_v19.min(),pred_v19.max())
print("첫 10개:",pred_v19[:10])

Prediction range: 0.0 1.0
첫 10개: [0.56 0.97 0.19 0.49 0.53 0.46 0.78 0.46 0.57 0.88]


## 8. 제출 파일

In [18]:
submission_v19=submission.copy()
submission_v19[ID_COLUMN]=test[ID_COLUMN]
submission_v19[TARGET]=pred_v19
output_path=OUTPUT_DIR/"submit_v19_pair_triple_neighbor.csv"
submission_v19.to_csv(output_path,index=False,encoding="utf-8")
print("Saved:",output_path)
submission_v19.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v19_pair_triple_neighbor.csv


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.49
4,TEST_0004,0.53
5,TEST_0005,0.46
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.57
9,TEST_0009,0.88


## 제출 기준

- `Wins: 3 / 3`
- `Worst split improvement >= 0.0001`
- `Selected 3D weight > 0`
- 세 조건을 모두 만족할 때만 수정 V19를 제출 후보로 검토
- 조건을 만족하지 못하면 Public 0.1272333의 2D 기준 모델 유지